In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [4]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [5]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [6]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [10]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### ANN

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [15]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [16]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [17]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [20]:
# Build our model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

In [23]:
model = ANN()

# loss, optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [25]:
# training the NN

epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # params update

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch + 1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.7154696713323179
epoch = 2/100, loss = 1.1283381995947466
epoch = 3/100, loss = 0.7680172583331233
epoch = 4/100, loss = 0.5630160518314528
epoch = 5/100, loss = 0.4553500569385031
epoch = 6/100, loss = 0.4010796896789385
epoch = 7/100, loss = 0.34655815622080927
epoch = 8/100, loss = 0.32393859845140704
epoch = 9/100, loss = 0.28981959431067755
epoch = 10/100, loss = 0.2676642352472181
epoch = 11/100, loss = 0.25218819340933923
epoch = 12/100, loss = 0.2348086869587069
epoch = 13/100, loss = 0.21723930440519168
epoch = 14/100, loss = 0.20947383052628973
epoch = 15/100, loss = 0.19749610061230866
epoch = 16/100, loss = 0.18487073221932288
epoch = 17/100, loss = 0.17821236813197966
epoch = 18/100, loss = 0.1705216431747312
epoch = 19/100, loss = 0.17046328698811325
epoch = 20/100, loss = 0.16393683866962142
epoch = 21/100, loss = 0.15759132154609845
epoch = 22/100, loss = 0.1491458880836549
epoch = 23/100, loss = 0.1460302285850048
epoch = 24/100, loss = 0.144029

In [26]:
# Evaluate

model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) 
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual samples in each batch

print("total vals: ", total)
print("correct vals: ", correct)
print("accuracy: ", correct/total)

total vals:  180
correct vals:  172
accuracy:  0.9555555555555556
